# Planning:


### Required Data:
Profiles can be nodes in a graph, each containing the necessary data. They will each have a user id that can be accessed through a dictionary.
Connections, blocked users, and skills can be stored under the profile node, in a set.
User id, name, job title, employer, industry, and location can simply be variables under each profile node.
A disconnected group, an isolated user, and at least one blocked pair can be created when generating the graph.

### Required Data Structures:
Dictionary will be used when storing / searching for user ids.
Sets will be used for connections, blocked users, and skills.
Adjacency List Graph will be used for the profiles and connections.
Queue can be used to implement breadth first search for if two users are connected.
Heap can be used to implement / store top connection recommendations
**Note, the required extension should use a data structure other than the above five in some justifiable way**




In [323]:
import random
import math

In [324]:
class Profile:

    def __init__(self, user_id, name, location, job_title = "Unemployed", employer = "None", industry = "None"):

        self.user_id = user_id
        self.name = name
        self.location = location
        self.job_title = job_title
        self.employer = employer
        self.industry = industry
        self.connections = set()
        self.blocked_users = set()
        self.skills = set()

    def __str__(self):

        return f"User #{self.user_id}: {self.name} lives in {self.location} and works as a {self.job_title} for {self.employer} in the {self.industry} industry.\nThey have the following skills: {str(self.skills)}"

    def connection_print(self):

        to_print = f"Name: {self.name}. Connections: | "

        for c in self.connections:
            to_print += f"{c.name} | "

        print(to_print)
        return None


In [325]:
#Added only for heap usage in recommendations
class HeapNode:

    def __init__(self, value, user, depth):

        self.value = value
        self.user = user
        self.left = None
        self.right = None
        self.parent = None
        self.max_depth = 0
        self.depth = depth

    def __str__(self):

        queue = [self]
        to_print = ""

        while len(queue) > 0:
            current = queue[0]
            to_print += f"\n{str(current.value)} "
            if current.left:
                to_print += f"with children {current.left.value} "
                queue.append(current.left)
                if current.right:
                    to_print += f"and {current.right.value}"
                    queue.append(current.right)
            queue.pop(0)

        return to_print

    def insert (self, new_value, new_user):

        def find_open_spot(self):

            if self.max_depth == 0:
                spot = (self,True)

            else:
                temp = self
                while temp.right:
                    temp = temp.right
                if temp.depth == self.max_depth:
                    candidate_depth = self.max_depth
                else:
                    candidate_depth = self.max_depth - 1
                candidates = []
                queue = [self]

                while len(queue) > 0:

                    current_node = queue[0]

                    if current_node.depth == candidate_depth:
                        candidates.append(current_node)

                    else:
                        if current_node.left:
                            queue.append(current_node.left)
                        if current_node.right:
                            queue.append(current_node.right)

                    queue.pop(0)


                for i in range(len(candidates)):

                    c = candidates[i]
                    if not (c.left and c.right):
                        if not c.left:
                            spot = (c,True)
                        else:
                            spot = (c,False)
                        break
            return spot

        #Swaps the node up the tree until its parent is larger or its parent is tied and the parent's ID is larger
        def swappify(just_added):

            current = just_added


            while current.parent.value < current.value:

                bigger = current.value
                bigger_user = current.user
                current.value = current.parent.value
                current.user = current.parent.user
                current.parent.value = bigger
                current.parent.user = bigger_user
                current = current.parent
                if current.parent == None:
                    break

            if current.parent:
                while current.parent.value == current.value and current.parent.user.user_id < current.user.user_id:

                    bigger_user = current.user
                    current.user = current.parent.user
                    current.parent.user = bigger_user
                    current = current.parent
                    if current.parent == None:
                        break

            return current

        def reconfigure_depth(node):

            current = node
            while current.parent:

                if current.parent.max_depth == current.max_depth:
                    current.parent.max_depth += 1
                current = current.parent
            return None
                

        spot = find_open_spot(self)

        new_parent = spot[0]
        new_depth = new_parent.depth + 1

        if spot[1] == True:
            new_parent.left = HeapNode(new_value,new_user,new_depth)
            new_parent.left.parent = new_parent
            new_parent.left.max_depth = 0

            reconfigure_depth(new_parent.left)

            newly_added = new_parent.left

        else:
            new_parent.right = HeapNode(new_value,new_user,new_depth)
            new_parent.right.parent = new_parent
            new_parent.right.max_depth = 0

            newly_added = new_parent.right

        final_position = swappify(newly_added)
        return final_position

    def extract_max(self):

        def find_last(self):

            current = self
            already_checked = set()

            while current.right:
                current = current.right

            while current.depth != self.max_depth:
                already_checked.add(current)
                while current in already_checked:
                    if current.parent.right == current:
                        current = current.parent.left
                        while current.right:
                            current = current.right
                        if current.left:
                            current = current.left
                    else:
                        current = current.parent

            return current

        #Swaps the node down the tree until both its children are smaller or one / both of its children are tied AND those children's IDs are smaller
        def swappify(self):

            current = self

            while current.value < current.left.value or current.value < current.right.value:
                if current.right.value > current.left.value:
                    smaller = current.value
                    smaller_user = current.user
                    current.value = current.right.value
                    current.user = current.right.user
                    current.right.value = smaller
                    current.right.user = smaller_user
                    current = current.right
                else:
                    smaller = current.value
                    smaller_user = current.user
                    current.value = current.left.value
                    current.user = current.left.user
                    current.left.value = smaller
                    current.left.user = smaller_user
                    current = current.left
                if current.right == None:
                    break

            if current.right == None:
                if current.left:
                    if current.left.value > current.value:
                        smaller = current.value
                        smaller_user = current.user
                        current.value = current.left.value
                        current.user = current.left.user
                        current.left.value = smaller
                        current.left.user = smaller_user
                        current = current.left

            if current.left and current.right:
                #While one of the children have the same value and that child has a larger ID
                while (current.value == current.left.value or current.value == current.right.value):

                    if current.value == current.left.value and current.value == current.right.value:
                        if current.user.user_id < current.right.user.user_id or current.user.user_id < current.left.user.user_id:
                            if current.right.user.user_id > current.left.user.user_id:
                                smaller_user = current.user
                                current.user = current.right.user
                                current.right.user = smaller_user
                                current = current.right
                            else:
                                smaller_user = current.user
                                current.user = current.left.user
                                current.left.user = smaller_user
                                current = current.left
                            if current.right == None:
                                break
                        else:
                            break

                    elif current.value == current.right.value:
                        if current.user.user_id < current.right.user.user_id:
                            smaller_user = current.user
                            current.user = current.right.user
                            current.right.user = smaller_user
                            current = current.right
                            if current.right == None:
                                break
                        else:
                            break

                    else:
                        if current.user.user_id < current.left.user.user_id:
                            smaller_user = current.user
                            current.user = current.right.user
                            current.right.user = smaller_user
                            current = current.right
                            if current.right == None:
                                break
                        else:
                            break

            if current.right == None:
                if current.left:
                    if current.user.user_id < current.left.user.user_id:
                        smaller_user = current.user
                        current.user = current.right.user
                        current.right.user = smaller_user
                        current = current.right
 
            return None

        max_value = self.value
        max_user = self.user

        last = find_last(self)
        self.value = last.value
        self.user = last.user

        if last.parent:
            if last.parent.right:
                last.parent.right = None
            else:
                last.parent.left = None

        swappify(self)

        return (max_value,max_user)

        

In [326]:
#Generic Functions
def shortest_connection_chain(start_user, target_user, print_it = False):

    if start_user == target_user:
        if print_it == True:
            print(f"That's the same user.")
        return((0,[start_user.name]))

    queue = []
    #Uses tuple to store the user Profile and the names of connections along the way
    current_node_path = (start_user,[start_user.name])
    queue.append(current_node_path)

    visited = set()

    found = False

    #Breadth First Search Loop
    while len(queue) != 0:

        #splits tuple into component parts for readability
        current_node = current_node_path[0]
        current_path = current_node_path[1]

        for c in current_node.connections:

            #This connection is the one we're looking for
            if c == target_user:
                current_path.append(c.name)
                found = True
                break

            #This connection needs to be searched later
            if c not in visited:
                new_path = current_path.copy()
                new_path.append(c.name)
                if (c,new_path) not in queue:
                    queue.append((c,new_path))

        #Prints / Returns the depth and path if we found the correct connection
        if found == True:
            depth = len(current_path) - 1
            if print_it == True:
                print(f"The shortest path between {start_user.name} and {target_user.name} is {str(depth)} connections. The path is {str(current_path)}")
            return((depth,current_path))

        #Only activates if the correct connection was not found. Remove from queue, add to visited, and change current_node_path
        visited.add(current_node)
        queue.pop(0)
        if len(queue) != 0:
            current_node_path = queue[0] 

    #We only get here if length of queue is 0, meaning no connection could be found
    if print_it == True:
        print(f"{start_user.name} and {target_user.name} are not connected.")
    return((None,None))

def connect(user,friends_list):

    for f in friends_list:
        user.connections.add(f)
        f.connections.add(user)

    return None

def id_name_print(dictionary):
    to_print = ""
    for id,user in dictionary.items():
        to_print += f"User #{str(id)} is {user.name}\n"
    print(to_print)
    return None

def find_network_components(user_dict):

    if not len(user_dict) > 0:
        print("Empty Data Set")
        return None

    user_ids = list(user_dict.keys())
    communities_output = []
    community_count = 0

    while len(user_ids) > 0:

        current_user_id = user_ids[0]
        queue = []
        visited = set()
        queue.append(current_user_id)
        
        community_count += 1
        names = []

        while len(queue) > 0:

            current_user = user_dict[current_user_id]
            print(current_user_id)
            user_ids.remove(current_user_id)

            names.append(current_user.name)
            visited.add(current_user_id)

            for c in current_user.connections:

                if c.user_id not in visited:

                    if c.user_id not in queue:
                        queue.append(c.user_id)

            queue.pop(0)
            if len(queue) > 0:
                current_user_id = queue[0]

        communities_output.append((community_count,len(names),names))

    for num, count, names_list in communities_output:

        if count > 1:
            print(f"Community number {str(num)} has {str(count)} people in it. They are:")
            for n in names_list:
                print(f"\t{n}")
        else:
            print(f"Community number {str(num)} is an isolated user named {names_list[0]}")

    return None

def top_connection_recommendations(user_id,user_dict,k):

    user = user_dict[user_id]
    heap = None

    candidate_ids = list(user_dict.keys())
    candidate_ids.remove(user_id)

    for c in user.connections:
        candidate_ids.remove(c.user_id)

    for b in user.blocked_users:
        candidate_ids.remove(b.user_id)

    for a in candidate_ids:

        candidate = user_dict[a]
        score = 0

        for c in user.connections:
            if c in candidate.connections:
                score += 4
                break

        for s in user.skills:
            if s in candidate.skills:
                score += 2
                break

        if user.industry == candidate.industry:
            score += 2

        if user.location == candidate.location:
            score += 1

        if heap:
            heap.insert(score,user_dict[a])
        else:
            heap = HeapNode(score,user_dict[a],0)

    print (f"{user.name}'s top {str(k)} recommendations:")

    for i in range(k):
        score_person = heap.extract_max()
        score = score_person[0]
        person = score_person[1]

        mutual_connections = 0
        mutual_skills = 0
        mutual_industry = False
        mutual_location = False

        for c in user.connections:
            if c in person.connections:
                mutual_connections += 1

        for s in user.skills:
            if s in person.skills:
                mutual_skills += 1

        if user.industry == person.industry:
            mutual_industry = True

        if user.location == person.location:
            mutual_industry = True

        to_print = f"Candidate #{str(i+1)}: {person.name} has {str(mutual_connections)} connections in common. "
        if mutual_skills > 0:
            to_print += f"They have {str(mutual_skills)} skills in common. "
        if mutual_industry == True:
            to_print += f"They both work in the {person.industry} industry. "
        if mutual_location == True:
            to_print += f"They both live in {person.location}. "
        to_print += f"Final Score: {str(score)}"

        print(to_print)
    return None

        
            
        
        


In [327]:
#Functions for generating dataset
def generate_profiles(number,user_dict):

    #Surnames chosen from most common surnames in the united states according to census.gov
    surnames = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Rodriguez","Davis","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson"]
    #Given names not chosen that way
    given_names = ["Tom","Tommy","Thomas","Thomathan","Thomicheal","Thomater","Thomeven","Thomephen","Thouglas","Thombert","Orion"]
    #Cities / Locations chosen from the most populous cities in the united states according to census.gov, with the word "city" removed (who calls it 'Chicago city')
    locations = ["New York","Los Angeles","Chicago","Houston","Phoenix"]
    #industries are tuples with the name of the industry and some example jobs in that industry
    industries = [("Construction",["Construction Worker","Construction Manager"]),("Accounting",["Tax Accountant","Auditer","Private Accountant"]),("Tech",["Data Scientist","Software Developer","Obnoxious CEO"])]
    employers = ["Hegemonic Monopoly Inc.","Small Startup and Co."]
    skills = ["Conflict Mitigation","Adaptability","Innovative Thinking","Communication","Strategic Thinking","Customer Service","Creativity","Client Management","Customer Retention","Relationship Management"]
   

    for i in range(number):

        user_id = i+1
        name = f"{random.choice(given_names)} {random.choice(surnames)}"
        location = random.choice(locations)
        industry_list = random.choice(industries)
        industry = industry_list[0]
        job_title = random.choice(industry_list[1])
        employer = random.choice(employers)
        skills_amount = random.randint(1,4)
        profile_skills = random.sample(skills,skills_amount)

        new_node = Profile(user_id,name,location,job_title,employer,industry)
        for skill in profile_skills:
             new_node.skills.add(skill)
        user_dict[user_id] = new_node

    return user_dict

def connect_the_dots(user_dict, minimum_connections = 35, isolated_users = 0, disconnected_triplets = 0):

    def isolate_them(ids):

            for i in range(isolated_users):
                isolated = random.choice(ids)
                ids.remove(isolated)
            return None

    def triplet_disconnection(ids):

        for k in range(disconnected_triplets):
        
                disconnected_ids = random.sample(ids,3)
        
                for y in disconnected_ids:
                    ids.remove(y)
                    friends = []
                    for x in disconnected_ids:
                         if x != y:
                              friends.append(user_dict[x])
                    connect(user_dict[y],friends)
                
        return None

    user_count = len(user_dict)
    minimum_users = isolated_users + (disconnected_triplets * 3)
    possible_connections = math.factorial(user_count - minimum_users) + (disconnected_triplets * 3)
    connection_count = 0

    #checking for errors
    if minimum_connections > possible_connections:
        raise Exception("Not possible to create the minimum number of connections")

    if user_count < minimum_users:
        raise Exception("Not enough users for that many isolated or disconnected groups!")

    user_ids = list(user_dict.keys())

    isolate_them(user_ids)

    triplet_disconnection(user_ids)

    connection_count += disconnected_triplets * 3

    #Stores the user_ids who are not isolated or in a triplet
    social_user_ids = user_ids.copy()

    #Creating a blocked pair
    blocked_pair = random.sample(social_user_ids,2)

    block1 = user_dict[blocked_pair[0]]
    block2 = user_dict[blocked_pair[1]]

    block1.blocked_users.add(block2)
    block2.blocked_users.add(block1)

    #For each id, makes a list of who can't be connected, then randomly assigns some connections
    for id in user_ids:

        profile = user_dict[id]
        cant_friend = {profile}

        for c in profile.connections:

             cant_friend.add(c)

        for c in profile.blocked_users:
             cant_friend.add(c)

        #Note: Potential Friends capped at 3 arbitrarily, to allow for less connections, but could be increased, especially for larger graphs
        number_of_friends = random.randint(0,3)
        friends_list = set()

        while len(friends_list) < number_of_friends:
             potential_friend = user_dict[random.choice(user_ids)]
             if potential_friend not in cant_friend:
                  friends_list.add(potential_friend)
                  cant_friend.add(potential_friend)
                  connection_count += 1
                  
        connect(profile,friends_list)

    #randomly makes connections until minimum connections is satisfied
    while connection_count < minimum_connections:
         
         lonely_node = user_dict[random.choice(social_user_ids)]

         max_connections = len(social_user_ids) - 1 - len(lonely_node.blocked_users)

         if max_connections > len(lonely_node.connections):

            cant_friend = {lonely_node}
            potential_friend = lonely_node

            for friend in lonely_node.connections:
                cant_friend.add(friend)

            for blocked in lonely_node.blocked_users:
                 cant_friend.add(blocked)

            while potential_friend in cant_friend:
                 potential_friend = user_dict[random.choice(social_user_ids)]

            connect(lonely_node,[potential_friend])
            connection_count += 1

    return None
            


In [328]:
user_dict = {}

generate_profiles(30,user_dict)
connect_the_dots(user_dict,35,3,1)

for user in user_dict.values():
    print (user)
    user.connection_print()

random_user = random.choice(list(user_dict.keys()))
top_connection_recommendations(random_user,user_dict,5)


User #1: Thombert Anderson lives in New York and works as a Construction Worker for Small Startup and Co. in the Construction industry.
They have the following skills: {'Adaptability'}
Name: Thombert Anderson. Connections: | Thomas Johnson | Thomas Martinez | 
User #2: Thombert Jones lives in Chicago and works as a Construction Worker for Hegemonic Monopoly Inc. in the Construction industry.
They have the following skills: {'Relationship Management'}
Name: Thombert Jones. Connections: | Thomas Miller | Orion Lopez | 
User #3: Thomas Johnson lives in Houston and works as a Auditer for Small Startup and Co. in the Accounting industry.
They have the following skills: {'Innovative Thinking', 'Relationship Management', 'Communication', 'Conflict Mitigation'}
Name: Thomas Johnson. Connections: | Thombert Anderson | Thomas Gonzalez | Tom Rodriguez | Tommy Martinez | Tom Hernandez | 
User #4: Thomeven Smith lives in New York and works as a Software Developer for Small Startup and Co. in the Te